##### Task 11
Build a customer summary table.

What I want in the end:
- One row per customer
- Columns including:
  - `customer_id`
  - `name`
  - `country`
  - number of valid orders
  - total items purchased
  - total revenue
  - average order value

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
-- Build a customer summary table with one row per customer

CREATE OR REPLACE TEMP VIEW enriched_order_lines AS
SELECT o.order_id, o.customer_id, i.item_id, o.quantity, i.price, o.quantity * i.price revenue
FROM (
    SELECT order_id, customer_id, item.item_id, item.quantity
    FROM silver_orders
    LATERAL VIEW explode(items) AS item
) o
LEFT JOIN silver_items i
    ON o.item_id = i.item_id;

CREATE OR REPLACE TEMP VIEW order_summary AS
SELECT 
    order_id, 
    customer_id, 
    SUM(revenue) order_revenue,
    SUM(quantity) total_quantity,
    COUNT(item_id) distinct_items
FROM enriched_order_lines
GROUP BY order_id, customer_id;

CREATE OR REPLACE TEMP VIEW customer_summary AS
SELECT 
    c.customer_id, 
    c.name, 
    c.country, 
    COUNT(o.order_id) valid_orders,
    SUM(o.total_quantity) total_items_purchased,
    SUM(o.order_revenue) total_revenue,
    AVG(o.order_revenue) average_order_value
FROM silver_customers c
LEFT JOIN order_summary o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.country
HAVING valid_orders > 0
ORDER BY valid_orders DESC;

SELECT *
FROM customer_summary;